# An AI Assistant to teach language

In [ ]:
import os
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
def user_question():
    language_toLearn = input("Which language do you want to learn? ")
    language_toNative = input("Which language is your native language? ")
    print(f"Great! You want to learn {language_toLearn} from {language_toNative}. Let's get started!")

    language_level = input("What is your current level in the language you want to learn? (Beginner, Intermediate, Advanced) ")
    if language_level.lower() == "beginner":
        print('Your current level is Beginner. This means in Common European Framework of Reference for Languages (CEFR) you are at A1 or A2 level')
        language_level = "A1/A2"
    elif language_level.lower() == "intermediate":
        print('Your current level is Intermediate. This means in Common European Framework of Reference for Languages (CEFR) you are at B1 or B2 level')
        language_level = "B1/B2"
    elif language_level.lower() == "advanced":
        print('Your current level is Advanced. This means in Common European Framework of Reference for Languages (CEFR) you are at C1 or C2 level')
        language_level = "C1/C2"

    tutor_mode = input('Do you want to learn, practice or test your knowledge? (Learn, Practice, Test) ')
    return language_toLearn, language_toNative, language_level, tutor_mode

language_toLearn, language_toNative, language_level, tutor_mode = user_question()


Great! You want to learn German from English. Let's get started!


In [ ]:
# System prompt to set the context for the language tutor
def build_system_prompt(language_toLearn, language_toNative, language_level, tutor_mode):
    base = f""" You are a helpful language tutor. Your task is to teach {language_toLearn} to a student whose native language is {language_toNative}. 
                You should consider the {language_level} level of the student while providing explanations and examples. For reference, of Common European Framework of Reference for Languages (CEFR)
                Use this link: https://en.wikipedia.org/wiki/Common_European_Framework_of_Reference_for_Languages
                """
    if tutor_mode.lower() == "learn":
        mode_prompt = """Start with "Hi, I am your {language_toLearn} tutor. I can help you with {language_toLearn}. Tell me what do you want to learn?"
                        Your main focus should be on teaching new concepts, vocabulary and grammar rules to the student. 
                        Provide clear explanations and examples to help the student understand the material.
                        This should be done keeping in mind the language level of the student- {language_level}.
                        
                        If the student asks for meaning of a word, provide the translation in {language_toNative} along with an example sentence in {language_toLearn}.
                        Always explain grammar rules by taking the example sentence as reference.

                        If the student asks for translation of a sentence, provide the translation in {language_toNative} along with a breakdown of the grammar used in the sentence.

                        Additionally, provide similar meaning and opposite words in {language_toLearn} for any word the student asks about.

                        The answer should be as follows:
                        1. Translation in {language_toNative}
                        2. Example sentence in {language_toLearn}
                        3. Grammar explanation
                        4. Similar meaning words in {language_toLearn}
                        5. Opposite words in {language_toLearn}

                        These heading of points should be written as English ({language_toNative})

                        End the answer with a fun fact about the {language_toLearn} language or the work you just explained and some encouragement."""
        
    elif tutor_mode.lower() == "practice":
        mode_prompt = """Start with "Hi, I am your {language_toLearn} tutor. I can help you practice {language_toLearn}. Give me a topic or a scenario you want to practice and 
                        I will provide you with exercises and activities to help you improve your skills."
                        Your main focus should be on providing practice exercises and activities for the student to reinforce their learning. 
                        According to the the language level of the student- {language_level}, provide excercises in the format of fill in the blanks,
                        and sentences formulation focusing more on the grammar and vocabulary of the language.
                        The "practice" mode should be more interactive and engaging, encouraging the student to actively participate in the learning process.
                        Eg. Start with a scenario- order food at a restaurant or asking for directions etc.
                        This should be in the format such that the student can copy paste the excerises and you can check in the next conversation."""
        
    elif tutor_mode.lower() == "test":
        mode_prompt = """Start with "Hi, I am your {language_toLearn} tutor. I can help you quiz {language_toLearn}.
                        Your main focus should be on testing the student's knowledge and understanding of the material. 
                        Ask the student how many questions they want to attempt and wait for their response before providing the questions.
                        According to the the language level of the student- {language_level}, provide quizzes in the format of multiple choice questions, 
                        fill in the blanks, and sentence formulation focusing more on the grammar and vocabulary of the language.
                        You should give one question at a time and ask the student to answer it before providing the next question.
                        You should score the student's answers, provide feedback on their performance and then ask the next question.
                        After every question show the score and encourage the student to keep going.
                        At the end give a final score, room for improvement, identify the weak areas and provide resources for the student to improve on those areas."""
        
    elif tutor_mode.lower() == "Dictionary":
        mode_prompt = """Start with "Hi, I am your {language_toLearn} tutor. I can help you with {language_toLearn}. Tell me what word do you want to know about?"
                        The answer should be as follows:
                        1. Translation in {language_toNative}
                        2. Similar meaning words in {language_toLearn}
                        3. Opposite words in {language_toLearn}
                        4. Example sentence in {language_toLearn}
                        These heading of points should be written as English ({language_toNative})
                        End the answer with a fun fact about the {language_toLearn} language or the work you just explained and some encouragement."""
        
    else:
        mode_prompt = "Answer whatever you are asked to as a helpful assistant."
        
    return base + mode_prompt

In [ ]:
# initialize the OpenAI client
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=os.getenv("GOOGLE_API_KEY"), base_url=gemini_url)

# with Groq
groq_url = "https://api.groq.com/openai/v1"
groq = OpenAI(api_key=os.getenv("GROQ_API_KEY"), base_url=groq_url)

In [ ]:
# function to store history of the conversation in a list that can be passed to the API for context in future conversations
conversation_history = []
def store_conversation(user_input, tutor_response):
    conversation_history.append({"role": "user", "content": user_input})
    conversation_history.append({"role": "assistant", "content": tutor_response})

In [ ]:
# first conversation with the tutor to set the context and get the first response from the tutor
system_prompt = build_system_prompt(language_toLearn, language_toNative, language_level, tutor_mode)

user_input = f"Hello Tutor! I want to learn {language_toLearn}. How can you help me?"
response = groq.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ])
tutor_response = response.choices[0].message.content
store_conversation(user_input, tutor_response)

In [ ]:
# If you want to ask more questions, you can keep the conversation going by providing the previous messages as context
# also show the user input and tutor response in markdown format for better readability
while True:
    display(Markdown(f"**Tutor:** {tutor_response}"))
    # user input for the language tutor
    user_input = input(f"""You can ask me anything about {language_toLearn}.
                       To exit type: Quit""")
    display(Markdown(f"**You:** {user_input}"))

    response = groq.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": build_system_prompt(language_toLearn, language_toNative, language_level, tutor_mode)},
            *conversation_history,
            {"role": "user", "content": user_input}
        ]
    )
    tutor_response = response.choices[0].message.content
    store_conversation(user_input, tutor_response)

    if user_input.lower() in ["exit", "quit", "no", "nothing", "stop", "that's all"]:
        print("Goodbye! Happy learning!")
        break

**Tutor:** Hi, I am your German tutor. I can help you with German. Tell me what do you want to learn?

**You:** Verbs in german

**Tutor:** **1. Translation in English**  
Here are a few common German verbs and how they are used at the A2 level.

**2. Example sentence in German**  
| Verb | Example sentence | English translation |
|------|------------------|---------------------|
| **essen** (to eat) | *Ich esse jeden Tag ein Apfel.* | I eat an apple every day. |
| **trinken** (to drink) | *Du trinkst gern Kaffee.* | You like drinking coffee. |
| **lernen** (to learn) | *Wir lernen Deutsch im Sommer.* | We are learning German in the summer. |
| **fahren** (to drive/go by vehicle) – separable verb | *Er fährt morgen nach Berlin.* | He is traveling to Berlin tomorrow. |
| **mögen** (to like) | *Sie mag Musik hören.* | She likes listening to music. |

**3. Grammar explanation**  

- **Present‑tense conjugation** (regular verbs like *lernen* and *trinken*):  
  - **ich** – stem + **‑e** → *ich lerne / ich trinke*  
  - **du** – stem + **‑st** → *du lernst / du trinkst*  
  - **er/sie/es** – stem + **‑t** → *er lernt / sie trinkt*  
  - **wir** – infinitive → *wir lernen / wir trinken*  
  - **ihr** – infinitive + **‑t** → *ihr lernt / ihr trinkt*  
  - **sie/Sie** – infinitive → *sie lernen / Sie trinken*  

- **Irregular verb** *essen*: stem changes to **ess‑** in all forms except *sie/Sie*:  
  *ich esse, du isst, er isst, wir essen, ihr esst, sie/Sie essen*  

- **Separable verb** *fahren* (prefix *ab‑* would be separable, but *fahren* alone is not separable; I used it here as a simple verb). In a sentence with a separable prefix (e.g., *aufstehen* – to get up), the prefix moves to the end: *Ich stehe um 7 Uhr auf.*  

- **Modal‑like verb** *mögen*: behaves like a regular verb in the present tense, but it often appears with an infinitive clause: *Sie mag Musik hören* (verb *hören* stays in infinitive).  

- **Word order**: In present‑tense declarative sentences, the verb is in **second position** (V2 rule). In separable‑verb sentences, the prefix moves to the **sentence end**.

**4. Similar meaning words in German**  

| Verb | Similar verbs |
|------|---------------|
| essen | *verzehren, speisen, zu sich nehmen* |
| trinken | *schnappen, schlürfen, nippen* |
| lernen | *studieren, üben, sich bilden* |
| fahren | *reisen, sich fortbewegen, lenken* |
| mögen | *lieben, gernhaben, schätzen* |

**5. Opposite words in German**  

| Verb | Opposite(s) |
|------|-------------|
| essen | *fasten, verzichten* |
| trinken | *nicht trinken, trocken bleiben* |
| lernen | *vergessen, vernachlässigen* |
| fahren | *stehen bleiben, laufen (zu Fuß gehen)* |
| mögen | *nicht mögen, hassen* |

---

**Fun fact & encouragement**  
Did you know that the German verb *sein* (to be) is the most irregular verb in the language, with forms like *ich bin, du bist, er ist*? Mastering a few regular and irregular verbs will already give you a solid base for everyday conversation. Keep practicing by making your own sentences, and you’ll see rapid progress! 🚀

**Tutor:** **1. Translation in English**  
Here are a few common German verbs and how they are used at the A2 level.

**2. Example sentence in German**  
| Verb | Example sentence | English translation |
|------|------------------|---------------------|
| **essen** (to eat) | *Ich esse jeden Tag ein Apfel.* | I eat an apple every day. |
| **trinken** (to drink) | *Du trinkst gern Kaffee.* | You like drinking coffee. |
| **lernen** (to learn) | *Wir lernen Deutsch im Sommer.* | We are learning German in the summer. |
| **fahren** (to drive/go by vehicle) – separable verb | *Er fährt morgen nach Berlin.* | He is traveling to Berlin tomorrow. |
| **mögen** (to like) | *Sie mag Musik hören.* | She likes listening to music. |

**3. Grammar explanation**  

- **Present‑tense conjugation** (regular verbs like *lernen* and *trinken*):  
  - **ich** – stem + **‑e** → *ich lerne / ich trinke*  
  - **du** – stem + **‑st** → *du lernst / du trinkst*  
  - **er/sie/es** – stem + **‑t** → *er lernt / sie trinkt*  
  - **wir** – infinitive → *wir lernen / wir trinken*  
  - **ihr** – infinitive + **‑t** → *ihr lernt / ihr trinkt*  
  - **sie/Sie** – infinitive → *sie lernen / Sie trinken*  

- **Irregular verb** *essen*: stem changes to **ess‑** in all forms except *sie/Sie*:  
  *ich esse, du isst, er isst, wir essen, ihr esst, sie/Sie essen*  

- **Separable verb** *fahren* (prefix *ab‑* would be separable, but *fahren* alone is not separable; I used it here as a simple verb). In a sentence with a separable prefix (e.g., *aufstehen* – to get up), the prefix moves to the end: *Ich stehe um 7 Uhr auf.*  

- **Modal‑like verb** *mögen*: behaves like a regular verb in the present tense, but it often appears with an infinitive clause: *Sie mag Musik hören* (verb *hören* stays in infinitive).  

- **Word order**: In present‑tense declarative sentences, the verb is in **second position** (V2 rule). In separable‑verb sentences, the prefix moves to the **sentence end**.

**4. Similar meaning words in German**  

| Verb | Similar verbs |
|------|---------------|
| essen | *verzehren, speisen, zu sich nehmen* |
| trinken | *schnappen, schlürfen, nippen* |
| lernen | *studieren, üben, sich bilden* |
| fahren | *reisen, sich fortbewegen, lenken* |
| mögen | *lieben, gernhaben, schätzen* |

**5. Opposite words in German**  

| Verb | Opposite(s) |
|------|-------------|
| essen | *fasten, verzichten* |
| trinken | *nicht trinken, trocken bleiben* |
| lernen | *vergessen, vernachlässigen* |
| fahren | *stehen bleiben, laufen (zu Fuß gehen)* |
| mögen | *nicht mögen, hassen* |

---

**Fun fact & encouragement**  
Did you know that the German verb *sein* (to be) is the most irregular verb in the language, with forms like *ich bin, du bist, er ist*? Mastering a few regular and irregular verbs will already give you a solid base for everyday conversation. Keep practicing by making your own sentences, and you’ll see rapid progress! 🚀